In [9]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

train_transaction = pd.read_csv('ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('ieee-fraud-detection/train_identity.csv')
test_transaction  = pd.read_csv('ieee-fraud-detection/test_transaction.csv')
test_identity     = pd.read_csv('ieee-fraud-detection/test_identity.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test  = test_transaction.merge(test_identity,  on='TransactionID', how='left')
test.columns = test.columns.str.replace('-', '_')

split = train['TransactionDT'].quantile(0.80)
tr  = train[train['TransactionDT'] <= split].copy()
val = train[train['TransactionDT'] >  split].copy()
print(f"tr {tr.shape} | val {val.shape} | test {test.shape}")

tr (472432, 434) | val (118108, 434) | test (506691, 433)


### Feature definition

In [10]:
delete_cols = ['id_29','V191','V196','V28','V241','V107','V117','V119','V120',
               'V113','V305','V88','V89','V27','V41','V65','V325','V327','V68','id_16','id_27']

cat_fea = ['ProductCD','card1','card2','card3','card4','card5','card6',
           'addr1','addr2','P_emaildomain','R_emaildomain','DeviceType','DeviceInfo']

for i in range(1, 10):
    if 'M'+str(i) not in delete_cols:
        cat_fea.append('M'+str(i))
for i in range(12, 39):
    if 'id_'+str(i) not in delete_cols:
        cat_fea.append('id_'+str(i))

num_fea = [f for f in train.columns
           if f not in ['TransactionID','isFraud'] + cat_fea + delete_cols]

### TransactionDT & TransactionAmt features

In [11]:
for df in [tr, val, test]:
    df['hour']          = (df['TransactionDT'] // 3600) % 24
    df['dayofweek']     = (df['TransactionDT'] // 86400) % 7
    df['day']           = df['TransactionDT']  // 86400
    df['hour_sin']      = np.sin(2 * np.pi * df['hour']      / 24)
    df['hour_cos']      = np.cos(2 * np.pi * df['hour']      / 24)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)

    df['log_amt']        = np.log1p(df['TransactionAmt'])
    df['amt_decimal']    = (df['TransactionAmt'] % 1).round(2).astype('category')
    df['amt_is_round']   = (df['TransactionAmt'] % 1 == 0).astype(np.int8)
    df['amt_last_digit'] = pd.Series(np.where(
        df['TransactionAmt'] % 1 == 0,
        df['TransactionAmt'].astype(int) % 10, -1
    ), index=df.index).astype('category')

num_fea.extend(['hour','dayofweek','day','hour_sin','hour_cos',
                'dayofweek_sin','dayofweek_cos','log_amt','amt_is_round'])
cat_fea.extend(['amt_decimal','amt_last_digit'])

### Log transform high-skew numeric features (|skew| > 10)

In [12]:
skew_series = tr[num_fea].select_dtypes(include=[np.number]).skew()
high_skew_cols = skew_series[skew_series.abs() > 10].index.tolist()
print(f"Log transforming {len(high_skew_cols)} features: {high_skew_cols}")

for col in high_skew_cols:
    for df in [tr, val, test]:
        df[col + "_log"] = np.log1p(df[col].clip(lower=0))
    num_fea.append(col + "_log")

            

Log transforming 195 features: ['TransactionAmt', 'C1', 'C2', 'C3', 'C4', 'C6', 'C7', 'C8', 'C10', 'C11', 'C12', 'C14', 'V1', 'V14', 'V23', 'V37', 'V38', 'V44', 'V45', 'V47', 'V55', 'V56', 'V77', 'V78', 'V86', 'V87', 'V96', 'V99', 'V100', 'V101', 'V102', 'V103', 'V104', 'V105', 'V106', 'V108', 'V109', 'V110', 'V111', 'V112', 'V114', 'V116', 'V118', 'V121', 'V122', 'V123', 'V125', 'V126', 'V127', 'V128', 'V129', 'V130', 'V131', 'V132', 'V133', 'V134', 'V135', 'V136', 'V137', 'V138', 'V142', 'V161', 'V162', 'V163', 'V166', 'V167', 'V168', 'V172', 'V176', 'V177', 'V178', 'V179', 'V180', 'V182', 'V183', 'V186', 'V187', 'V190', 'V192', 'V193', 'V198', 'V199', 'V200', 'V201', 'V202', 'V203', 'V204', 'V205', 'V206', 'V207', 'V208', 'V209', 'V210', 'V211', 'V212', 'V213', 'V214', 'V215', 'V216', 'V218', 'V219', 'V220', 'V221', 'V222', 'V223', 'V224', 'V225', 'V226', 'V227', 'V228', 'V229', 'V230', 'V231', 'V232', 'V236', 'V240', 'V242', 'V243', 'V244', 'V245', 'V246', 'V247', 'V248', 'V249', '

In [13]:
len(num_fea)

569

### Bin D15

In [14]:
_, bin_edges = pd.qcut(tr['D15'], q=10, retbins=True, duplicates='drop')
for df in [tr, val, test]:
    df['D15_bin'] = pd.cut(df['D15'], bins=bin_edges, labels=False,
                           include_lowest=True).astype('category')
cat_fea.append('D15_bin')

features = num_fea + cat_fea
print(f"Total features: {len(features)} | numeric: {len(num_fea)} | categorical: {len(cat_fea)}")

Total features: 618 | numeric: 569 | categorical: 49


### 24h rolling fraud features per entity

In [15]:
# def rolling_fraud_stats(base_df, target_df, col, window=86400, exclude_self=False):
#     base_s   = base_df[[col, "TransactionDT", "isFraud"]].sort_values("TransactionDT").reset_index(drop=True)
#     target_s = target_df[[col, "TransactionDT"]].reset_index(drop=True).copy()
#     target_s["_pos"] = np.arange(len(target_s))

#     tx_counts    = np.zeros(len(target_s), dtype=np.int32)
#     fraud_counts = np.zeros(len(target_s), dtype=np.int32)

#     for entity, grp in target_s.groupby(col):
#         dt_target  = grp["TransactionDT"].values
#         base_grp   = base_s[base_s[col] == entity]
#         dt_base    = base_grp["TransactionDT"].values
#         fraud_base = base_grp["isFraud"].values
#         if len(dt_base) == 0:
#             continue
#         left  = np.searchsorted(dt_base, dt_target - window, side="left")
#         upper = dt_target - 1 if exclude_self else dt_target
#         right = np.searchsorted(dt_base, upper, side="right")
#         pos = grp["_pos"].values
#         tx_counts[pos]    = right - left
#         fraud_counts[pos] = np.array([fraud_base[l:r].sum() for l, r in zip(left, right)])

#     fraud_ratio = np.where(tx_counts > 0, fraud_counts / tx_counts, np.nan)
#     return tx_counts, fraud_counts, fraud_ratio

# entities = ["card1", "card2", "addr1", "P_emaildomain"]

# # sort and reset index — use these sorted frames to build X directly (no mapping back)
# tr_s   = tr.sort_values("TransactionDT").reset_index(drop=True)
# val_s  = val.sort_values("TransactionDT").reset_index(drop=True)
# test_s = test.sort_values("TransactionDT").reset_index(drop=True)

# rolling_features = []
# for col in entities:
#     print(f"Computing {col}...")
#     _, fc, fr = rolling_fraud_stats(tr_s, tr_s, col, exclude_self=True)
#     tr_s[col + "_fraud_count_24h"] = fc
#     tr_s[col + "_fraud_ratio_24h"] = fr

#     _, fc, fr = rolling_fraud_stats(tr_s, val_s, col, exclude_self=False)
#     val_s[col + "_fraud_count_24h"] = fc
#     val_s[col + "_fraud_ratio_24h"] = fr

#     _, fc, fr = rolling_fraud_stats(tr_s, test_s, col, exclude_self=False)
#     test_s[col + "_fraud_count_24h"] = fc
#     test_s[col + "_fraud_ratio_24h"] = fr

#     rolling_features += [col + "_fraud_count_24h", col + "_fraud_ratio_24h"]

# num_fea.extend(rolling_features)
# features = num_fea + cat_fea
# print(f"Added {len(rolling_features)} rolling features. Total: {len(features)}")

### Categorical encoding

In [16]:
n = 255
for col in list(cat_fea):
    cur_unique = tr[col].nunique()
    if cur_unique <= n:
        for df in [tr, val, test]:
            df[col] = df[col].astype("category")
    else:
        freq_map = tr[col].value_counts()
        for df in [tr, val, test]:
            df[col + '_freq'] = df[col].map(freq_map)
        num_fea.extend([col + '_freq'])
                
        # le = LabelEncoder()
        # tr_vals = tr[col].astype(str)
        # le.fit(list(tr_vals) + ["unknown"])
        # known = set(le.classes_)
        # tr[col]   = le.transform(tr_vals)
        # val[col]  = le.transform([v if v in known else "unknown" for v in val[col].astype(str)])
        # test[col] = le.transform([v if v in known else "unknown" for v in test[col].astype(str)])
        # le_dict[col] = le
        cat_fea.remove(col)   # remove from cat_fea — freq encoded as numeric
        # num_fea.extend([col])

features = cat_fea + num_fea
print(f"cat_fea: {len(cat_fea)} | num_fea: {len(num_fea)}")


cat_fea: 41 | num_fea: 577


In [17]:
from sklearn.preprocessing import LabelEncoder

X_tr   = tr[features].copy()
X_val  = val[features].copy()
X_test = test[features].copy()
y_tr   = tr["isFraud"]
y_val  = val["isFraud"]

# low_card_cat  = []   # native categorical — LightGBM handles grouping
# high_card_cat = []   # label encoded as numeric — avoids 255 bin limit

# le_dict = {}
# for col in cat_fea:
#     if col not in features:
#         continue
#     n_unique = X_tr[col].nunique()
#     if n_unique <= 255:
#         # native: fix categories from tr so val/test are consistent
#         tr_cats = X_tr[col].astype("category").cat.categories
#         X_tr[col]   = pd.Categorical(X_tr[col],  categories=tr_cats)
#         X_val[col]  = pd.Categorical(X_val[col],  categories=tr_cats)
#         X_test[col] = pd.Categorical(X_test[col], categories=tr_cats)
#         low_card_cat.append(col)
#     else:
#         # label encode as integer so LightGBM treats as numeric
#         le = LabelEncoder()
#         tr_vals = X_tr[col].astype(str)
#         le.fit(list(tr_vals) + ["unknown"])
#         known = set(le.classes_)
#         X_tr[col]   = le.transform(tr_vals)
#         X_val[col]  = le.transform([v if v in known else "unknown" for v in X_val[col].astype(str)])
#         X_test[col] = le.transform([v if v in known else "unknown" for v in X_test[col].astype(str)])
#         le_dict[col] = le
#         high_card_cat.append(col)

# print(f"Native categorical ({len(low_card_cat)}): {low_card_cat}")
# print(f"Label encoded     ({len(high_card_cat)}): {high_card_cat}")
# print(f"X_tr shape: {X_tr.shape}")

model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=256,
    random_state=42,
    n_jobs=-1
)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    categorical_feature=cat_fea,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.242014 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 57173
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 618
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312784
[LightGBM] [Info] Start training from score -3.312784
Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.0878829
[200]	valid_0's binary_logloss: 0.0869004
Early stopping, best iteration is:
[178]	valid_0's binary_logloss: 0.0866988


,boosting_type,'gbdt'
,num_leaves,256
,max_depth,-1
,learning_rate,0.05
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


### Validation performance

In [19]:
val_pred = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, val_pred)
print(f"Baseline AUC : 0.9208")
print(f"Current  AUC : {auc:.4f}")
print(f"Improvement  : {auc - 0.9208:+.4f}")

Baseline AUC : 0.9208
Current  AUC : 0.9206
Improvement  : -0.0002


### Feature importance

In [20]:
importance = (pd.DataFrame({
    'feature':    features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True))

print("=== Top 20 ===")
print(importance.head(20).to_string(index=False))
print("=== New engineered features ===")
new_feats = ['hour','dayofweek','day','hour_sin','hour_cos','dayofweek_sin','dayofweek_cos',
             'log_amt','amt_is_round','amt_decimal','amt_last_digit','D15_bin']
print(importance[importance['feature'].isin(new_feats)].to_string(index=False))

=== Top 20 ===
       feature  importance
   amt_decimal        2713
    card1_freq        2649
    addr1_freq        2309
    card2_freq        2261
TransactionAmt        1966
 TransactionDT        1923
         dist1         987
           D15         923
           C13         923
         id_31         867
            D4         776
           D10         750
 P_emaildomain         716
         card5         687
            D2         675
            C1         589
            D1         569
         id_02         565
      hour_sin         536
          hour         512
=== New engineered features ===
       feature  importance
   amt_decimal        2713
      hour_sin         536
          hour         512
           day         463
      hour_cos         459
     dayofweek         304
 dayofweek_cos         233
 dayofweek_sin         227
amt_last_digit         165
       D15_bin         117
  amt_is_round          51
       log_amt           0


### Generate test predictions & submission

In [22]:
# predict on test
test_pred = model.predict_proba(X_test)[:, 1]

# TransactionID must align with test_s (sorted by TransactionDT)
submission = pd.DataFrame({
    'TransactionID': test['TransactionID'].values,
    'isFraud':       test_pred
})

# sort back to original test order (Kaggle expects original order)
test_original_order = test_transaction['TransactionID'].values
submission = submission.set_index('TransactionID').loc[test_original_order].reset_index()

submission.to_csv('submission_v2_remove_freq_2.csv', index=False)
print(f"Submission saved: {submission.shape}")
print(submission.head())
print(f"isFraud prediction stats:")
print(submission['isFraud'].describe().round(4))

Submission saved: (506691, 2)
   TransactionID   isFraud
0        3663549  0.000606
1        3663550  0.002518
2        3663551  0.001532
3        3663552  0.002199
4        3663553  0.003994
isFraud prediction stats:
count    506691.0000
mean          0.0299
std           0.1176
min           0.0001
25%           0.0019
50%           0.0041
75%           0.0109
max           0.9989
Name: isFraud, dtype: float64
